In [ ]:
import json
import os
from datetime import date

DATA_FILE = "attendance.json"
MIN_PERCENT = 75  # minimum attendance required


def load_data():
    if os.path.exists(DATA_FILE):
        with open(DATA_FILE, "r") as f:
            return json.load(f)
    return {}  # {student_name: {"2026-09-21": "P", ...}}


def save_data(data):
    with open(DATA_FILE, "w") as f:
        json.dump(data, f, indent=2)


def add_student(data):
    name = input("Student name: ").strip().title()
    if not name:
        print("Name can't be empty.")
    elif name in data:
        print(f"{name} already exists.")
    else:
        data[name] = {}
        save_data(data)
        print(f"Added {name}.")


def mark_attendance(data):
    if not data:
        print("No students yet. Add some first.")
        return

    day = input("Date (YYYY-MM-DD, press Enter for today): ").strip()
    if not day:
        day = str(date.today())
    else:
        try:
            date.fromisoformat(day)
        except ValueError:
            print("Invalid date format.")
            return

    for name in data:
        while True:
            status = input(f"{name} - Present or Absent? (p/a): ").strip().upper()
            if status in ("P", "A"):
                data[name][day] = status
                break
            print("Please type p or a.")

    save_data(data)
    print(f"Attendance saved for {day}.")


def percentage(records):
    if not records:
        return 0.0
    present = sum(1 for s in records.values() if s == "P")
    return present / len(records) * 100


def show_report(data):
    if not data:
        print("No data yet.")
        return

    print(f"\n{'Name':<20}{'Present':<10}{'Total':<8}{'%':<8}Status")
    print("-" * 55)
    for name, records in data.items():
        present = sum(1 for s in records.values() if s == "P")
        total = len(records)
        pct = percentage(records)
        status = "OK" if pct >= MIN_PERCENT else "LOW"
        print(f"{name:<20}{present:<10}{total:<8}{pct:<8.1f}{status}")


def student_history(data):
    name = input("Student name: ").strip().title()
    if name not in data:
        print("Student not found.")
        return
    if not data[name]:
        print("No records yet.")
        return
    for day in sorted(data[name]):
        label = "Present" if data[name][day] == "P" else "Absent"
        print(f"{day}: {label}")


def remove_student(data):
    name = input("Student to remove: ").strip().title()
    if name in data:
        del data[name]
        save_data(data)
        print(f"Removed {name}.")
    else:
        print("Student not found.")


def main():
    data = load_data()
    actions = {
        "1": ("Add student", add_student),
        "2": ("Mark attendance", mark_attendance),
        "3": ("Attendance report", show_report),
        "4": ("Student history", student_history),
        "5": ("Remove student", remove_student),
    }

    while True:
        print("\n=== Attendance Tracker ===")
        for key, (label, _) in actions.items():
            print(f"{key}. {label}")
        print("6. Exit")

        choice = input("Choose: ").strip()
        if choice == "6":
            print("Bye!")
            break
        elif choice in actions:
            actions[choice][1](data)
        else:
            print("Invalid choice.")


if __name__ == "__main__":
    main()


=== Attendance Tracker ===
1. Add student
2. Mark attendance
3. Attendance report
4. Student history
5. Remove student
6. Exit
